# 4. Train Custom Reranker
Huấn luyện reranker custom trên dữ liệu Vietnamese

**Phiên bản này sử dụng `Trainer.train()` (HuggingFace Transformers) thay vì `model.fit()`, với đầy đủ:**
- 📉 Hàm loss (BCE)
- 📈 Learning rate schedule (warmup + cosine decay)
- 🔢 Thông số mô hình (parameter count)
- 📊 Learning curve (train/val loss theo bước)
- 🏆 Kết quả đánh giá từng thành phần pipeline

## 4.0 Chuẩn bị Colab

In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✅ Drive mounted")

In [ ]:
# Install requirements
%pip install -q bitsandbytes accelerate
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"
%pip install -q datasets scikit-learn
%pip install -q faiss-cpu
%pip install -q ragas langchain-groq
%pip install -q transformers>=4.36.0 sentence-transformers

In [ ]:
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

## 4.1 Chuẩn bị dữ liệu training

In [ ]:
import os
import gc
import json
import random
import pickle
import hashlib
import logging
import jsonlines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset as TorchDataset

# HuggingFace Transformers – dùng cho Trainer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CrossEncoderClassificationEvaluator

import sys
sys.path.append('.')
from src.data.data_loader import DataLoader as CustomDataLoader

data_loader = CustomDataLoader("data")
print("✅ Imports & Setup successful")

In [ ]:
DATASETS = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]

all_train_data = []
for ds in DATASETS:
    try:
        data = data_loader.load_dataset(ds, "train")
        all_train_data.extend(data)
        print(f"✅ {ds}: {len(data)} samples")
    except Exception as e:
        print(f"⚠️  {ds}: {e}")

print(f"\n📊 Total: {len(all_train_data)} samples")

In [ ]:
PAIRS_CACHE = Path("data/reranker_training_data.jsonl")

In [ ]:
def extract_fields(sample: dict):
    query = sample.get("query") or sample.get("question") or ""
    context = sample.get("context") or sample.get("passage") or sample.get("text") or ""
    if context.startswith("http"):
        context = sample.get("ground_truth") or sample.get("answer") or ""
    return query.strip(), context.strip()

if PAIRS_CACHE.exists():
    with jsonlines.open(str(PAIRS_CACHE)) as r:
        all_pairs = list(r)
    positive_pairs = [p for p in all_pairs if p["label"] == 1]
    negative_pairs = [p for p in all_pairs if p["label"] == 0]
    print(f"✅ Loaded cached pairs: {len(all_pairs)} total ({len(positive_pairs)} pos / {len(negative_pairs)} neg)")
    SKIP_PAIR_BUILD = True
else:
    positive_pairs = []
    for s in all_train_data:
        q, ctx = extract_fields(s)
        if q and ctx and not ctx.startswith("http"):
            positive_pairs.append({"query": q, "document": ctx, "label": 1})
    print(f"✅ {len(positive_pairs)} positive pairs")
    SKIP_PAIR_BUILD = False

In [ ]:
def extract_eval_fields(sample: dict):
    query   = sample.get("query") or sample.get("question") or ""
    context = sample.get("context") or sample.get("passage") or sample.get("text") or ""
    gt = sample.get("ground_truth") or sample.get("answer") or ""
    if not gt:
        answers = sample.get("answers", {})
        if isinstance(answers, dict):
            texts = answers.get("text", [])
            gt = texts[0] if texts else ""
        elif isinstance(answers, list) and answers:
            gt = answers[0] if isinstance(answers[0], str) else ""
    return query.strip(), gt.strip(), context.strip()

In [ ]:
if not SKIP_PAIR_BUILD:
    from rank_bm25 import BM25Okapi

    all_docs = list(set([
        extract_fields(s)[1] for s in all_train_data
        if extract_fields(s)[1] and not extract_fields(s)[1].startswith("http")
    ]))
    sample_pool = positive_pairs

    tokenized_docs = [d.lower().split() for d in all_docs]
    bm25 = BM25Okapi(tokenized_docs)

    negative_pairs = []
    for pp in tqdm(sample_pool, desc="Building hard negatives"):
        scores = bm25.get_scores(pp["query"].lower().split())
        top_indices = np.argsort(scores)[::-1][:20]
        hard_neg = None
        for idx in top_indices:
            if all_docs[idx] != pp["document"]:
                hard_neg = all_docs[idx]
                break
        if hard_neg is None:
            hard_neg = random.choice(all_docs)
        negative_pairs.append({"query": pp["query"], "document": hard_neg, "label": 0})

    if len(negative_pairs) > len(positive_pairs):
        negative_pairs = random.sample(negative_pairs, len(positive_pairs))

    all_pairs = positive_pairs + negative_pairs
    random.shuffle(all_pairs)

    PAIRS_CACHE.parent.mkdir(parents=True, exist_ok=True)
    with jsonlines.open(PAIRS_CACHE, mode="w") as writer:
        writer.write_all(all_pairs)
    print(f"✅ Saved: {PAIRS_CACHE}  ({PAIRS_CACHE.stat().st_size / 1024:.0f} KB)")
    print(f"✅ Positive: {len(positive_pairs)}  |  Negative: {len(negative_pairs)}")
    print(f"📊 Total pairs: {len(all_pairs)}")

## 4.2 Train Cross-Encoder với `Trainer.train()` (HuggingFace Transformers)

### Lý do chuyển từ `model.fit()` → `Trainer.train()`:
| Tiêu chí | `CrossEncoder.fit()` | `Trainer.train()` |
|---|---|---|
| Logging loss/lr chi tiết | Hạn chế | ✅ Đầy đủ theo step |
| Custom callback | Khó | ✅ Dễ dàng |
| Mixed precision (fp16/bf16) | Không có | ✅ Có |
| Gradient checkpointing | Không có | ✅ Có |
| Resume từ checkpoint | Không có | ✅ Có |
| Learning curve visualisation | Phải tự code | ✅ Có qua `trainer.state.log_history` |

In [ ]:
import gc, os
import torch

gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
print(f"GPU RAM sau cleanup: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")

In [ ]:
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "BAAI/bge-reranker-base"
OUTPUT_DIR = Path("models/reranker")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINETUNED_MODEL = OUTPUT_DIR / "final"

print(f"Device      : {DEVICE}")
print(f"Base model  : {BASE_MODEL}")
print(f"Output dir  : {OUTPUT_DIR}")

In [ ]:
# ─── Load tokenizer & model ───────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
hf_model  = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
    trust_remote_code=True,
    ignore_mismatched_sizes=True,
)
print("✅ Tokenizer & model loaded")

In [ ]:
import re

def remap_state_dict(state_dict):
    """Đổi tên gamma→weight, beta→bias trong LayerNorm để tương thích transformers mới."""
    new_sd = {}
    for k, v in state_dict.items():
        k_new = re.sub(r'\.gamma$', '.weight', k)
        k_new = re.sub(r'\.beta$',  '.bias',   k_new)
        new_sd[k_new] = v
    return new_sd

In [ ]:
# ─── Hiển thị số lượng tham số mô hình ────────────────────────────────────────
def count_parameters(model):
    total   = sum(p.numel() for p in model.parameters())
    train   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen  = total - train
    return total, train, frozen

total_params, train_params, frozen_params = count_parameters(hf_model)

print("\n" + "═"*55)
print("      📦  THÔNG SỐ MÔ HÌNH (Model Parameters)")
print("═"*55)
print(f"  Tổng tham số          : {total_params:>15,}")
print(f"  Tham số trainable     : {train_params:>15,}")
print(f"  Tham số frozen        : {frozen_params:>15,}")
print(f"  Kích thước ước tính   : {total_params * 4 / 1024**2:>13.1f} MB  (fp32)")
print(f"  Kích thước ước tính   : {total_params * 2 / 1024**2:>13.1f} MB  (fp16)")
print("═"*55)

# Chi tiết từng layer lớn
print("\n📋 Top layers theo số tham số:")
layer_params = []
for name, module in hf_model.named_modules():
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params > 0:
        layer_params.append((name or "root", params))

layer_params.sort(key=lambda x: x[1], reverse=True)
print(f"{'Layer':<45} {'Params':>12}")
print("-" * 58)
for name, params in layer_params[:15]:
    bar = "█" * int(params / total_params * 30)
    print(f"  {name:<43} {params:>10,}  {bar}")

In [ ]:
# ─── Chuẩn bị dataset theo dạng HuggingFace Dataset ──────────────────────────
TRAIN_SIZE = 20000
VAL_SIZE   = 2000
MAX_LEN    = 512

train_pairs, val_pairs_split = train_test_split(
    all_pairs, test_size=0.1, random_state=42
)
if len(train_pairs)     > TRAIN_SIZE: train_pairs     = random.sample(train_pairs,     TRAIN_SIZE)
if len(val_pairs_split) > VAL_SIZE:   val_pairs_split = random.sample(val_pairs_split, VAL_SIZE)

print(f"Train pairs : {len(train_pairs):,}")
print(f"Val pairs   : {len(val_pairs_split):,}")


# Tokenize với padding + truncation
def tokenize_batch(pairs, tokenizer, max_length=MAX_LEN):
    queries = [p["query"]    for p in pairs]
    docs    = [p["document"] for p in pairs]
    labels  = [float(p["label"]) for p in pairs]
    enc = tokenizer(
        queries, docs,
        padding=True, truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    enc["labels"] = torch.tensor(labels, dtype=torch.float)
    return enc


class RerankerDataset(TorchDataset):
    """Torch Dataset cho cross-encoder reranker."""
    def __init__(self, pairs, tokenizer, max_length=MAX_LEN):
        self.pairs     = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p   = self.pairs[idx]
        enc = self.tokenizer(
            p["query"], p["document"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(p["label"], dtype=torch.float),
        }


train_dataset = RerankerDataset(train_pairs, tokenizer)
val_dataset   = RerankerDataset(val_pairs_split, tokenizer)
print(f"✅ Datasets created — train: {len(train_dataset)}, val: {len(val_dataset)}")

In [ ]:
# ─── Custom Trainer với BCE loss ──────────────────────────────────────────────
class BCETrainer(Trainer):
    """
    Ghi đè compute_loss để dùng Binary Cross-Entropy Loss.
    Phù hợp với cross-encoder có num_labels=1 (relevance score).
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits.squeeze(-1)       # shape: (batch,)
        loss_fn = nn.BCEWithLogitsLoss()           # tự động sigmoid + BCE
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def _load_from_checkpoint(self, resume_from_checkpoint, model=None):
        import os, torch
        if model is None:
            model = self.model

        weights_file = os.path.join(resume_from_checkpoint, "pytorch_model.bin")
        safe_file    = os.path.join(resume_from_checkpoint, "model.safetensors")

        if os.path.exists(safe_file):
            from safetensors.torch import load_file
            state_dict = load_file(safe_file, device="cpu")
        elif os.path.exists(weights_file):
            state_dict = torch.load(weights_file, map_location="cpu")
        else:
            # Fallback về super() nếu không tìm thấy file
            return super()._load_from_checkpoint(resume_from_checkpoint, model)

        state_dict = remap_state_dict(state_dict)
        model.load_state_dict(state_dict, strict=False)
        print(f"✅ Checkpoint loaded + key remapped từ: {resume_from_checkpoint}")

In [ ]:
# ─── TrainingArguments – cấu hình đầy đủ ─────────────────────────────────────
NUM_EPOCHS    = 3
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1      # 10% steps đầu làm warmup
WEIGHT_DECAY  = 0.01
STEPS_PER_EPOCH = len(train_dataset) // BATCH_SIZE
TOTAL_STEPS     = STEPS_PER_EPOCH * NUM_EPOCHS

print("\n" + "═"*55)
print("      ⚙️   CẤU HÌNH HUẤN LUYỆN (Training Config)")
print("═"*55)
print(f"  Epochs              : {NUM_EPOCHS}")
print(f"  Batch size          : {BATCH_SIZE}")
print(f"  Learning rate       : {LEARNING_RATE}")
print(f"  LR scheduler        : cosine với linear warmup")
print(f"  Warmup ratio        : {WARMUP_RATIO*100:.0f}% ({int(TOTAL_STEPS*WARMUP_RATIO)} steps)")
print(f"  Weight decay        : {WEIGHT_DECAY}")
print(f"  Steps/epoch         : {STEPS_PER_EPOCH}")
print(f"  Total steps         : {TOTAL_STEPS}")
print(f"  Loss function       : BCEWithLogitsLoss")
print(f"  Precision           : fp16 (nếu GPU hỗ trợ)")
print("═"*55)


training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",          # cosine decay sau warmup
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    eval_strategy               = "steps",
    eval_steps                  = STEPS_PER_EPOCH // 4,   # eval 4 lần/epoch
    save_strategy               = "steps",
    save_steps                  = STEPS_PER_EPOCH // 4,   # eval 4 lần/epoch
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    logging_dir                 = str(OUTPUT_DIR / "logs"),
    logging_steps               = 10,                # log mỗi 10 bước
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    report_to                   = "none",
    seed                        = 42,
    save_total_limit            = 1
)
print("✅ TrainingArguments configured")

In [ ]:
# ─── Visualise Learning Rate Schedule TRƯỚC khi train ─────────────────────────
from transformers import get_cosine_schedule_with_warmup
import numpy as np
import matplotlib.pyplot as plt

warmup_steps = int(TOTAL_STEPS * WARMUP_RATIO)
_steps = np.arange(TOTAL_STEPS + 1)

def cosine_lr_with_warmup(step, warmup, total, max_lr):
    if step < warmup:
        return max_lr * step / max(1, warmup)
    progress = (step - warmup) / max(1, total - warmup)
    return max_lr * 0.5 * (1 + np.cos(np.pi * progress))

_lrs = [cosine_lr_with_warmup(s, warmup_steps, TOTAL_STEPS, LEARNING_RATE) for s in _steps]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(_steps, _lrs, color="#2196F3", linewidth=2.5, label="Learning rate")
ax.axvline(warmup_steps, color="orange", ls="--", lw=1.5, label=f"End warmup (step {warmup_steps})")

# Đánh dấu epoch boundaries
for ep in range(1, NUM_EPOCHS + 1):
    step_ep = ep * STEPS_PER_EPOCH
    ax.axvline(step_ep, color="gray", ls=":", lw=1, alpha=0.7)
    ax.text(step_ep - 5, max(_lrs) * 0.05, f"Epoch {ep}", ha="right", fontsize=9, color="gray")

ax.set_xlabel("Training Step", fontsize=12)
ax.set_ylabel("Learning Rate", fontsize=12)
ax.set_title("📈 Learning Rate Schedule (Cosine + Linear Warmup)", fontsize=13, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "lr_schedule.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ LR schedule saved")

In [ ]:
# ─── Tạo Trainer & bắt đầu huấn luyện ────────────────────────────────────────
if FINETUNED_MODEL.exists():
    print(f"✅ Tìm thấy model đã fine-tuned tại {FINETUNED_MODEL}, bỏ qua training")
    hf_model = AutoModelForSequenceClassification.from_pretrained(
        str(FINETUNED_MODEL), num_labels=1, trust_remote_code=True,
    )
    SKIP_TRAINING = True
else:
    print(f"⚠️  Chưa có model, bắt đầu training từ {BASE_MODEL}")
    SKIP_TRAINING = False

if not SKIP_TRAINING:
    gc.collect()
    torch.cuda.empty_cache()

    trainer = BCETrainer(
        model           = hf_model,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = val_dataset,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
    )

    print("\n🚀 Bắt đầu trainer.train() ...")
    train_result = trainer.train()     # ← ĐÂY LÀ TRAINER.TRAIN()

    print(f"\n✅ Training hoàn tất!")
    print(f"   Total steps       : {train_result.global_step}")
    print(f"   Training loss     : {train_result.training_loss:.4f}")
    print(f"   Training time     : {train_result.metrics['train_runtime']:.1f}s")
    print(f"   Samples/sec       : {train_result.metrics['train_samples_per_second']:.1f}")

    # Lưu model tốt nhất
    trainer.save_model(str(FINETUNED_MODEL))
    tokenizer.save_pretrained(str(FINETUNED_MODEL))
    print(f"\n💾 Model saved to {FINETUNED_MODEL}")
else:
    print("Skipping training — dùng model đã fine-tuned")

In [ ]:
# ─── Vẽ Learning Curve từ log_history ─────────────────────────────────────────
if not SKIP_TRAINING:
    log_history = trainer.state.log_history

    # Tách train loss và eval loss
    train_steps  = [h["step"]  for h in log_history if "loss"      in h and "eval_loss" not in h]
    train_losses = [h["loss"]  for h in log_history if "loss"      in h and "eval_loss" not in h]
    train_lrs    = [h["learning_rate"] for h in log_history if "learning_rate" in h]

    eval_steps   = [h["step"]      for h in log_history if "eval_loss" in h]
    eval_losses  = [h["eval_loss"] for h in log_history if "eval_loss" in h]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Panel 1: Loss curve ──
    ax1 = axes[0]
    ax1.plot(train_steps, train_losses, color="#2196F3", alpha=0.6, linewidth=1.5, label="Train loss")
    # Smoothed train loss
    if len(train_losses) > 10:
        window = max(5, len(train_losses) // 20)
        smooth  = np.convolve(train_losses, np.ones(window)/window, mode='valid')
        s_steps = train_steps[window-1:]
        ax1.plot(s_steps, smooth, color="#0D47A1", linewidth=2.5, label=f"Train loss (smooth, w={window})")
    ax1.plot(eval_steps, eval_losses, "o-", color="#F44336", linewidth=2, markersize=6, label="Val loss")

    # Đánh dấu epoch boundaries
    for ep in range(1, NUM_EPOCHS + 1):
        ax1.axvline(ep * STEPS_PER_EPOCH, color="gray", ls=":", lw=1, alpha=0.6)
        ax1.text(ep * STEPS_PER_EPOCH - 2, max(train_losses) * 0.98,
                 f"E{ep}", ha="right", fontsize=8, color="gray")

    ax1.set_xlabel("Training Step", fontsize=12)
    ax1.set_ylabel("Loss (BCE)", fontsize=12)
    ax1.set_title("📉 Learning Curve (Train vs Val Loss)", fontsize=12, fontweight="bold")
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # ── Panel 2: LR curve (thực tế từ log) ──
    ax2 = axes[1]
    lr_steps_actual = [h["step"] for h in log_history if "learning_rate" in h]
    lr_vals_actual  = [h["learning_rate"] for h in log_history if "learning_rate" in h]
    ax2.plot(lr_steps_actual, lr_vals_actual, color="#4CAF50", linewidth=2.5)
    ax2.set_xlabel("Training Step", fontsize=12)
    ax2.set_ylabel("Learning Rate", fontsize=12)
    ax2.set_title("📈 Learning Rate (thực tế trong training)", fontsize=12, fontweight="bold")
    ax2.yaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
    ax2.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
    ax2.grid(True, alpha=0.3)

    plt.suptitle("Learning Curve & LR Schedule — Cross-Encoder Reranker",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / "learning_curve.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ Learning curve saved")

## 4.3 Đánh giá: Fine-tuned vs Pre-trained

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score as sk_f1

# Load lại fine-tuned model qua CrossEncoder để dùng .predict()
model      = CrossEncoder(str(FINETUNED_MODEL), num_labels=1, device=DEVICE, trust_remote_code=True)
pretrained = CrossEncoder(BASE_MODEL,           num_labels=1, device=DEVICE, trust_remote_code=True)

val_s1  = [p["query"]    for p in val_pairs_split]
val_s2  = [p["document"] for p in val_pairs_split]
val_lbl = [int(p["label"]) for p in val_pairs_split]

print(f"✅ Models loaded | Val samples: {len(val_s1)}")

In [ ]:
def evaluate_model(ce_model, s1_list, s2_list, true_labels):
    raw   = ce_model.predict(list(zip(s1_list, s2_list)), show_progress_bar=True)
    probs = 1.0 / (1.0 + np.exp(-raw))
    preds = (probs >= 0.5).astype(int)
    true  = np.array(true_labels, dtype=int)
    return {
        "accuracy" : accuracy_score(true, preds),
        "precision": precision_score(true, preds, zero_division=0),
        "recall"   : recall_score(true, preds, zero_division=0),
        "f1"       : sk_f1(true, preds, zero_division=0),
    }, probs


def evaluate_ranking(ce_model, s1_list, s2_list, true_labels, k=10):
    from sklearn.metrics import roc_auc_score, average_precision_score
    raw   = ce_model.predict(list(zip(s1_list, s2_list)), show_progress_bar=False)
    probs = 1.0 / (1.0 + np.exp(-raw))
    true  = np.array(true_labels, dtype=int)
    return {
        "AUC-ROC"  : roc_auc_score(true, probs),
        "AP"       : average_precision_score(true, probs),
    }


print("Đang đánh giá Fine-tuned model...")
ft_metrics, ft_probs = evaluate_model(model,      val_s1, val_s2, val_lbl)
ft_ranking           = evaluate_ranking(model,      val_s1, val_s2, val_lbl)

print("Đang đánh giá Pre-trained model...")
pt_metrics, pt_probs = evaluate_model(pretrained,  val_s1, val_s2, val_lbl)
pt_ranking           = evaluate_ranking(pretrained, val_s1, val_s2, val_lbl)

In [ ]:
# ─── Bảng kết quả phân loại ───────────────────────────────────────────────────
all_metric_keys = list(ft_metrics.keys()) + list(ft_ranking.keys())

print("\n" + "═"*60)
print("   📊 KẾT QUẢ ĐÁNH GIÁ: Fine-tuned vs Pre-trained")
print("═"*60)
print(f"{'Metric':<14} {'Pre-trained':>12} {'Fine-tuned':>12} {'Δ':>10}")
print("-" * 50)
for k in ft_metrics:
    pt, ft = pt_metrics[k], ft_metrics[k]
    arrow  = "▲" if ft > pt else ("▼" if ft < pt else "─")
    print(f"  {k:<12} {pt:>12.4f} {ft:>12.4f} {arrow}{abs(ft-pt):>+8.4f}")
print("-" * 50)
for k in ft_ranking:
    pt, ft = pt_ranking[k], ft_ranking[k]
    arrow  = "▲" if ft > pt else ("▼" if ft < pt else "─")
    print(f"  {k:<12} {pt:>12.4f} {ft:>12.4f} {arrow}{abs(ft-pt):>+8.4f}")
print("═"*60)

In [ ]:
# ─── Visualise kết quả so sánh ────────────────────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Bar chart so sánh metrics ──
ax1     = axes[0]
metrics = list(ft_metrics.keys())
x       = np.arange(len(metrics))
w       = 0.35
ax1.bar(x - w/2, [pt_metrics[m] for m in metrics], w, label="Pre-trained", color="#90CAF9", edgecolor="#1565C0")
ax1.bar(x + w/2, [ft_metrics[m] for m in metrics], w, label="Fine-tuned",  color="#A5D6A7", edgecolor="#2E7D32")
ax1.set_xticks(x)
ax1.set_xticklabels([m.capitalize() for m in metrics], fontsize=11)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel("Score", fontsize=12)
ax1.set_title("Classification Metrics", fontsize=12, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(True, axis="y", alpha=0.3)
for bar in ax1.patches:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

# ── Panel 2: ROC Curve ──
ax2 = axes[1]
true = np.array(val_lbl)
for probs, label, color in [(pt_probs, "Pre-trained", "#2196F3"), (ft_probs, "Fine-tuned", "#4CAF50")]:
    fpr, tpr, _ = roc_curve(true, probs)
    from sklearn.metrics import auc
    roc_auc = auc(fpr, tpr)
    ax2.plot(fpr, tpr, linewidth=2, label=f"{label} (AUC={roc_auc:.3f})", color=color)
ax2.plot([0,1],[0,1], "k--", alpha=0.4, label="Random")
ax2.set_xlabel("False Positive Rate", fontsize=12)
ax2.set_ylabel("True Positive Rate", fontsize=12)
ax2.set_title("ROC Curve", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ── Panel 3: Score distribution ──
ax3 = axes[2]
ft_pos = ft_probs[true == 1]
ft_neg = ft_probs[true == 0]
ax3.hist(ft_neg, bins=30, alpha=0.6, color="#F44336", label="Negative (FT)", density=True)
ax3.hist(ft_pos, bins=30, alpha=0.6, color="#4CAF50", label="Positive (FT)", density=True)
ax3.axvline(0.5, color="black", ls="--", lw=1.5, label="Threshold=0.5")
ax3.set_xlabel("Predicted Probability", fontsize=12)
ax3.set_ylabel("Density", fontsize=12)
ax3.set_title("Score Distribution (Fine-tuned)", fontsize=12, fontweight="bold")
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

plt.suptitle("📊 So sánh Pre-trained vs Fine-tuned Reranker",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "model_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─── Kiểm tra mẫu dự đoán ────────────────────────────────────────────────────
print("\n🔍 Kiểm tra kết quả dự đoán trên mẫu:")

sample_indices = [0, 10, 20]
for i in sample_indices:
    query      = val_s1[i]
    document   = val_s2[i]
    true_label = val_lbl[i]

    ft_raw  = model.predict([(query, document)])[0]
    ft_prob = 1.0 / (1.0 + np.exp(-ft_raw))
    pt_raw  = pretrained.predict([(query, document)])[0]
    pt_prob = 1.0 / (1.0 + np.exp(-pt_raw))

    ft_pred = "✅ Relevant" if ft_prob >= 0.5 else "❌ Not relevant"
    correct = "✓" if (ft_prob >= 0.5) == bool(true_label) else "✗"

    print(f"\n{'─'*60}")
    print(f"  Mẫu #{i+1} | True label: {'Relevant' if true_label else 'Not relevant'} | FT prediction: {correct}")
    print(f"  Query    : {query[:90]}...")
    print(f"  Document : {document[:90]}...")
    print(f"  FT prob  : {ft_prob:.4f} → {ft_pred}")
    print(f"  PT prob  : {pt_prob:.4f} → {'✅ Relevant' if pt_prob >= 0.5 else '❌ Not relevant'}")

## 4.4 Đánh giá end-to-end với RAGAS (LLM-as-a-judge) + 3 cấu hình pipeline

In [ ]:
import copy, logging
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from langchain_groq import ChatGroq
from datasets import Dataset as HFDataset
from rouge_score import rouge_scorer as rs
from sentence_transformers import CrossEncoder

from src.retrieval.hybrid_retriever import HybridRetriever
from src.reranking.rank_rag import RankReranker, ContextSelector
from src.rewriting.query_rewriter import QueryRewriter
from src.generation.llm_generator import LLMGenerator
from evaluation.evaluator import RAGEvaluator

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

In [ ]:
def get_query(sample):
    for key in ("query", "question", "input", "prompt"):
        val = sample.get(key)
        if val and isinstance(val, str) and val.strip():
            return val.strip()
    return ""

def get_context_for_index(sample):
    for key in ("context", "passage", "text", "document", "content"):
        val = sample.get(key)
        if val and isinstance(val, str) and val.strip() and not val.strip().startswith("http"):
            return val.strip()
    for key in ("ground_truth", "answer"):
        val = sample.get(key)
        if val and isinstance(val, str) and val.strip() and not val.strip().startswith("http"):
            return val.strip()
    return ""

def get_ground_truth(sample, dataset_name=""):
    if dataset_name == "uit_viquad2":
        answers = sample.get("answers", {})
        if isinstance(answers, dict):
            texts = answers.get("text", [])
            if texts and isinstance(texts, list):
                gt = texts[0]
                if isinstance(gt, str) and gt.strip(): return gt.strip()
    for key in ("ground_truth", "answer", "label", "output", "response"):
        val = sample.get(key)
        if not val: continue
        if isinstance(val, list) and val: val = val[0]
        if isinstance(val, str) and val.strip() and not val.strip().startswith("http"):
            return val.strip()
    logger.warning(f"[{dataset_name}] Không tìm được ground_truth. Keys: {list(sample.keys())}")
    return ""

def get_eval_triple(sample, dataset_name):
    return (get_query(sample), get_ground_truth(sample, dataset_name), get_context_for_index(sample))

In [ ]:
def recall_hit(gt, ctx_texts, threshold=0.5):
    if not gt: return 0.0
    gt_tokens = set(gt.lower().split())
    if not gt_tokens: return 0.0
    for t in ctx_texts:
        t_tokens = set(t.lower().split())
        if len(gt_tokens & t_tokens) / len(gt_tokens) >= threshold:
            return 1.0
    return 0.0

def recall_hit_for_reranker(truth, doc_text):
    if not truth or not doc_text: return False
    gt_tokens = set(truth.lower().split())
    if not gt_tokens: return False
    d_tokens = set(doc_text.lower().split())
    return len(gt_tokens & d_tokens) / len(gt_tokens) >= 0.5

def get_ground_truth(sample, dataset_name=""):
    if dataset_name == "uit_viquad2":
        answers = sample.get("answers", {})
        if isinstance(answers, dict):
            texts = answers.get("text", [])
            if texts and isinstance(texts, list):
                gt = texts[0]
                if isinstance(gt, str) and gt.strip(): return gt.strip()
        # ← THÊM: fallback khi answers đã bị flatten
        elif isinstance(answers, list) and answers:
            gt = answers[0] if isinstance(answers[0], str) else ""
            if gt.strip(): return gt.strip()
        elif isinstance(answers, str) and answers.strip():
            return answers.strip()

    for key in ("ground_truth", "answer", "label", "output", "response"):
        val = sample.get(key)
        if not val: continue
        if isinstance(val, list) and val: val = val[0]
        if isinstance(val, str) and val.strip() and not val.strip().startswith("http"):
            return val.strip()

    logger.warning(
        f"[{dataset_name}] Không tìm được ground_truth. Keys: {list(sample.keys())}; "
        f"answers={sample.get('answers')!r}"  # ← THÊM: log giá trị thực tế
    )
    return ""

In [ ]:
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings

class GroqSingleGen(ChatGroq):
    def _create_chat_result(self, response, generation_info=None):
        return super()._create_chat_result(response, generation_info)

    def bind(self, **kwargs):
        kwargs.pop("n", None)
        return super().bind(**kwargs)

groq_llm           = GroqSingleGen(api_key=GROQ_API_KEY, model="llama-3.1-8b-instant")
ragas_llm          = LangchainLLMWrapper(groq_llm)
ragas_emb   = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
)
llm_gen            = LLMGenerator(provider="groq", groq_api_key=GROQ_API_KEY, groq_model="llama-3.1-8b-instant")
rewriter           = QueryRewriter(llm_client=llm_gen)
reranker           = RankReranker(method="cross-encoder", cross_encoder_model=str(FINETUNED_MODEL))
pretrained_reranker = CrossEncoder(BASE_MODEL, num_labels=1, device=DEVICE, trust_remote_code=True)
base_eval          = RAGEvaluator()
base_eval.rouge_scorer = rs.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

DATASETS_TO_EVAL = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]
SAMPLES_PER_DS   = 10
BATCH_SIZE       = 32
CACHE_DIR        = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

In [ ]:
def get_or_build_retriever(ds_name, docs_pool):
    cache_path = CACHE_DIR / f"{ds_name}/corpus.pkl"
    pool_hash  = hashlib.md5(" ".join(sorted(docs_pool)).encode()).hexdigest()[:8]
    hash_file  = cache_path.with_suffix(".hash")
    if cache_path.exists() and hash_file.exists() and hash_file.read_text() == pool_hash:
        with open(cache_path, "rb") as f:
            print(f"✅ Loaded cached index for {ds_name}")
            return pickle.load(f)
    retriever = HybridRetriever()
    retriever.build_index(docs_pool)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "wb") as f:
        pickle.dump(retriever, f)
    hash_file.write_text(pool_hash)
    print(f"💾 Built and cached index for {ds_name}")
    return retriever

def run_pipeline(query, retriever, use_rewrite=True, use_rerank=True, top_k=5):
    queries  = rewriter.rewrite(query) if use_rewrite else [query]
    raw_docs = retriever.retrieve(queries, top_k=10)
    if use_rerank:
        ranked = reranker.rerank(query, copy.deepcopy(raw_docs), top_k=10)
        docs   = ContextSelector(top_k=top_k, deduplicate=True).select(ranked)
    else:
        docs = raw_docs[:top_k]
    answer = llm_gen.generate(query, docs, max_tokens=300)
    if not answer:
        logger.warning(f"Empty answer for query: {query!r}")
    return answer, docs

from ragas import RunConfig

run_cfg = RunConfig(
    timeout=120,       # giây chờ tối đa mỗi lần gọi LLM
    max_retries=3,     # số lần retry khi lỗi
    max_wait=60,       # giây chờ tối đa giữa các lần retry
)

def ragas_eval(rows):
    empty = {"faithfulness": 0.0, "answer_relevancy": 0.0, "context_precision": 0.0}
    if not rows: return empty
    valid_rows = [r for r in rows if r.get("answer") and r.get("reference") and r.get("contexts")]
    if not valid_rows: return empty
    try:
        result = evaluate(
            HFDataset.from_list(valid_rows),
            metrics=[faithfulness, answer_relevancy, context_precision],
            llm=ragas_llm,
            embeddings=ragas_emb,
            run_config=run_cfg,   # ← THÊM
            raise_exceptions=True,
        )
        return result
    except Exception as e:
        print(f"  ❌ RAGAS eval FAILED: {e}")
        return empty

def mean(lst): return float(np.mean(lst)) if lst else 0.0

In [ ]:
tag_configs = [
    ("full",       True,  True),
    ("no_rewrite", False, True),
    ("no_rerank",  True,  False),
]

all_results_by_ds = {
    "em_scores":      {},
    "f1_scores":      {},
    "recall_k":       {},
    "pt_recall_list": {},
    "ft_recall_list": {},
    "pt_mrr_list":    {},
    "ft_mrr_list":    {},
    "rows_data": {tag: {} for tag, *_ in tag_configs},
}

In [ ]:
for ds in DATASETS_TO_EVAL:
    print(f"\n{'─'*60}")
    print(f"📂 Dataset: {ds.upper()}")
    print(f"{'─'*60}")
    try:
        for key in ("em_scores", "f1_scores", "recall_k"):
            all_results_by_ds[key][ds] = {tag: [] for tag, *_ in tag_configs}
        for key in ("pt_recall_list", "ft_recall_list", "pt_mrr_list", "ft_mrr_list"):
            all_results_by_ds[key][ds] = []
        for tag, *_ in tag_configs:
            all_results_by_ds["rows_data"][tag][ds] = []

        test_samples  = data_loader.load_dataset(ds, "test")[:SAMPLES_PER_DS]
        train_samples = data_loader.load_dataset(ds, "train")[:1000]

        docs_pool = list(set([
            get_context_for_index(s) for s in train_samples
            if get_context_for_index(s)
        ]))
        print(f"  📚 Docs pool size: {len(docs_pool)}")
        retriever = get_or_build_retriever(ds, docs_pool)

        all_query_doc_pairs = []
        sample_info = []
        skipped = 0

        for sample in tqdm(test_samples, desc=f"{ds} — eval"):
            q, gt, _ = get_eval_triple(sample, ds)
            if not q or not gt:
                skipped += 1
                continue

            for tag, use_rw, use_rr in tag_configs:
                ans, docs = run_pipeline(q, retriever, use_rw, use_rr)
                ctx_texts = [d["text"] for d in docs]
                m = base_eval.calculate_all(q, ans, docs, gt)
                all_results_by_ds["em_scores"][ds][tag].append(m["em"])
                all_results_by_ds["f1_scores"][ds][tag].append(m["f1"])
                all_results_by_ds["recall_k"][ds][tag].append(recall_hit(gt, ctx_texts))
                all_results_by_ds["rows_data"][tag][ds].append({
                    "question":  q, "answer": ans or "",
                    "contexts":  ctx_texts, "reference": gt,
                })

            raw_docs = retriever.retrieve([q], top_k=10)
            if raw_docs:
                start_idx = len(all_query_doc_pairs)
                for d in raw_docs:
                    all_query_doc_pairs.append([q, d["text"]])
                sample_info.append({"true_ctx": gt, "start_idx": start_idx,
                                    "end_idx": len(all_query_doc_pairs), "docs": raw_docs})

        if skipped:
            print(f"  ⚠️  Bỏ qua {skipped}/{len(test_samples)} samples")

        if all_query_doc_pairs:
            pt_scores = pretrained_reranker.predict(all_query_doc_pairs, batch_size=BATCH_SIZE, show_progress_bar=False)
            ft_scores = model.predict(           all_query_doc_pairs, batch_size=BATCH_SIZE, show_progress_bar=False)

            def get_ranked_docs(scores, docs):
                return [d for _, d in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)]

            def calc_retrieval_metrics(ranked, truth, k=5):
                top_k_texts = [d["text"] for d in ranked[:k]]
                recall = recall_hit(truth, top_k_texts)
                mrr    = next((1.0/(r+1) for r, d in enumerate(ranked)
                               if recall_hit_for_reranker(truth, d["text"])), 0.0)
                return recall, mrr

            for info in sample_info:
                s_pt = pt_scores[info["start_idx"]:info["end_idx"]]
                s_ft = ft_scores[info["start_idx"]:info["end_idx"]]
                r_pt, m_pt = calc_retrieval_metrics(get_ranked_docs(s_pt, info["docs"]), info["true_ctx"])
                r_ft, m_ft = calc_retrieval_metrics(get_ranked_docs(s_ft, info["docs"]), info["true_ctx"])
                all_results_by_ds["pt_recall_list"][ds].append(r_pt)
                all_results_by_ds["pt_mrr_list"][ds].append(m_pt)
                all_results_by_ds["ft_recall_list"][ds].append(r_ft)
                all_results_by_ds["ft_mrr_list"][ds].append(m_ft)

        print(f"✅ Xong dataset: {ds}")

    except Exception as e:
        import traceback
        print(f"❌ Lỗi dataset {ds}: {e}")
        traceback.print_exc()

In [ ]:
# ─── RAGAS + Kết quả từng dataset ─────────────────────────────────────────────
print(f"\n\n{'═'*80}")
print("Ƀ RUNNING RAGAS EVALUATION (LLM-as-a-judge)...")
print(f"{'═'*80}")

all_summary_e2e      = {}
all_summary_reranker = {}

for ds_name in DATASETS_TO_EVAL:
    print(f"\n{'═'*80}")
    print(f"Ƀ KẾT QUẢ CHO DATASET: {ds_name.upper()}")
    print(f"{'═'*80}")

    ds_em     = all_results_by_ds["em_scores"][ds_name]
    ds_f1     = all_results_by_ds["f1_scores"][ds_name]
    ds_recall = all_results_by_ds["recall_k"][ds_name]

    print("  ⏳ Chạy RAGAS...")
    res_full  = ragas_eval(all_results_by_ds["rows_data"]["full"][ds_name])
    res_no_rw = ragas_eval(all_results_by_ds["rows_data"]["no_rewrite"][ds_name])
    res_no_rr = ragas_eval(all_results_by_ds["rows_data"]["no_rerank"][ds_name])

    summary_e2e = pd.DataFrame([
        {"System": "Full Pipeline",      "EM": f"{mean(ds_em['full']):.4f}",
         "F1": f"{mean(ds_f1['full']):.4f}",   "Recall@5": f"{mean(ds_recall['full']):.4f}",
         "Faithfulness": f"{res_full.get('faithfulness', 0.0):.4f}",
         "Answer Relevance": f"{res_full.get('answer_relevancy', 0.0):.4f}",
         "Context Precision": f"{res_full.get('context_precision', 0.0):.4f}"},
        {"System": "No Query Rewriting", "EM": f"{mean(ds_em['no_rewrite']):.4f}",
         "F1": f"{mean(ds_f1['no_rewrite']):.4f}", "Recall@5": f"{mean(ds_recall['no_rewrite']):.4f}",
         "Faithfulness": f"{res_no_rw.get('faithfulness', 0.0):.4f}",
         "Answer Relevance": f"{res_no_rw.get('answer_relevancy', 0.0):.4f}",
         "Context Precision": f"{res_no_rw.get('context_precision', 0.0):.4f}"},
        {"System": "No Re-ranking",      "EM": f"{mean(ds_em['no_rerank']):.4f}",
         "F1": f"{mean(ds_f1['no_rerank']):.4f}",  "Recall@5": f"{mean(ds_recall['no_rerank']):.4f}",
         "Faithfulness": f"{res_no_rr.get('faithfulness', 0.0):.4f}",
         "Answer Relevance": f"{res_no_rr.get('answer_relevancy', 0.0):.4f}",
         "Context Precision": f"{res_no_rr.get('context_precision', 0.0):.4f}"},
    ]).set_index("System")

    print("\nƀ BẢNG 1: ĐÁNH GIÁ END-TO-END")
    print(summary_e2e.to_markdown())
    all_summary_e2e[ds_name] = summary_e2e

    pt_r = all_results_by_ds["pt_recall_list"][ds_name]
    ft_r = all_results_by_ds["ft_recall_list"][ds_name]
    pt_m = all_results_by_ds["pt_mrr_list"][ds_name]
    ft_m = all_results_by_ds["ft_mrr_list"][ds_name]

    summary_reranker = pd.DataFrame([
        {"Reranker": "Pre-trained", "Recall@5": f"{mean(pt_r):.4f}", "MRR": f"{mean(pt_m):.4f}"},
        {"Reranker": "Fine-tuned",  "Recall@5": f"{mean(ft_r):.4f}", "MRR": f"{mean(ft_m):.4f}"},
    ]).set_index("Reranker")

    print("\nƀ BẢNG 2: SO SÁNH PRE-TRAINED vs FINE-TUNED RERANKER")
    print(summary_reranker.to_markdown())
    all_summary_reranker[ds_name] = summary_reranker
    print("═" * 60)

In [ ]:
# ─── Tổng hợp kết quả tất cả datasets ────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

metric_cols = ["F1", "Recall@5", "Faithfulness", "Context Precision"]
systems     = ["Full Pipeline", "No Query Rewriting", "No Re-ranking"]
colors      = ["#4CAF50", "#2196F3", "#FF9800"]

for mi, metric in enumerate(metric_cols):
    ax = axes[mi]
    ds_names = list(all_summary_e2e.keys())
    x        = np.arange(len(ds_names))
    w        = 0.25

    for si, (sys, color) in enumerate(zip(systems, colors)):
        vals = [float(all_summary_e2e[ds].loc[sys, metric]) for ds in ds_names]
        bars = ax.bar(x + (si - 1) * w, vals, w, label=sys, color=color, alpha=0.85, edgecolor="white")
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels([d.upper() for d in ds_names], fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score", fontsize=11)
    ax.set_title(f"{metric}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("🏆 Kết Quả Đánh Giá End-to-End Theo Từng Dataset và Cấu Hình Pipeline",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "e2e_results.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ End-to-end result chart saved")

In [ ]:
# ─── Tổng hợp Reranker so sánh PT vs FT ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ds_names = list(all_summary_reranker.keys())
x = np.arange(len(ds_names))
w = 0.35

for ax_idx, metric in enumerate(["Recall@5", "MRR"]):
    ax = axes[ax_idx]
    pt_vals = [float(all_summary_reranker[ds].loc["Pre-trained", metric]) for ds in ds_names]
    ft_vals = [float(all_summary_reranker[ds].loc["Fine-tuned",  metric]) for ds in ds_names]

    ax.bar(x - w/2, pt_vals, w, label="Pre-trained", color="#90CAF9", edgecolor="#1565C0")
    ax.bar(x + w/2, ft_vals, w, label="Fine-tuned",  color="#A5D6A7", edgecolor="#2E7D32")

    for i, (pt, ft) in enumerate(zip(pt_vals, ft_vals)):
        ax.text(i - w/2, pt + 0.01, f"{pt:.3f}", ha="center", fontsize=8)
        ax.text(i + w/2, ft + 0.01, f"{ft:.3f}", ha="center", fontsize=8)
        delta = ft - pt
        color_d = "#2E7D32" if delta > 0 else "#C62828"
        ax.text(i, max(pt, ft) + 0.06, f"Δ{delta:+.3f}", ha="center", fontsize=8,
                color=color_d, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels([d.upper() for d in ds_names], fontsize=10)
    ax.set_ylim(0, 1.2)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title(f"Reranker {metric}: Pre-trained vs Fine-tuned", fontsize=12, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("🔍 So Sánh Pre-trained vs Fine-tuned Reranker Theo Dataset",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "reranker_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Reranker comparison chart saved")